# Семинар 2. Когорты, LTV и юнит-экономика в сегментах

**Вопрос недели:** сколько мы зарабатываем на заказе и на покупателе — и почему среднее врёт?

Четыре части, ~20 минут каждая:
1. **Когорты и удержание** — когортная матрица, недозревшие ячейки, удержание по каналам.
2. **Провал когорты «Чёрной пятницы»** — найти и объяснить механизм, а не просто «когорта плохая».
3. **Юнит «в среднем → в сегментах»** — LTV12, CAC, LTV/CAC и payback по каналам; решение по CPA-сетям разворачивается.
4. **Простой прогноз LTV и стресс-тест «CAC +20%»** — модель вшиваем в таблицу экономики, а не оптимизируем.

**Треки.** 🟢 *Базовый* — запускайте готовые ячейки и отвечайте на вопросы. 🔵 *Глубокий* — вопросы и ячейки со значком 🔵: сначала напишите сами, потом сверьтесь.

**Данные:** `mart.users` (80 000 новых пользователей, регистрации с марта 2025 по август 2026), `mart.orders` (заказ с деньгами и CM), `mart.activity` (пользователь × день с сессией).

**Договорённости о деньгах.**
- **CM заказа** — колонка `cm`: доход платформы минус логистика, промо, эквайринг и возвраты. Рекламы продавцов в `mart.orders` нет; добавляем её как **1,1% GMV** (паспорт, §3).
- **LTV12** — маржа (CM + реклама) покупателя за **365 дней с первого заказа**. Считаем только у когорт, которые прожили полный год.
- **CAC** по каналам — из паспорта «Лукошко», §8 (внутренние данные маркетинга).

In [ ]:
import pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve().parents[1] / "практика"   # папка ноутбука → корень репозитория → практика

import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect(str(ROOT / "data" / "lukoshko.duckdb"), read_only=True)

users = con.execute("SELECT * FROM mart.users").fetchdf()
orders = con.execute("SELECT * FROM mart.orders").fetchdf()
END = orders["order_ts"].max().normalize() + pd.Timedelta(days=1)   # первый день без данных
ADS = 0.011                                                         # реклама продавцов, доля GMV
print(f"пользователей {len(users):,}, заказов {len(orders):,}, данные до {END:%d.%m.%Y} (не включительно)")

## Разминка (5 мин). Юнит заказа на данных

Паспорт (§4): средний чек 1 850 ₽ → доход платформы 333 ₽ → минус логистика, промо, эквайринг → **≈ 167 ₽ CM** (9% GMV). Сверим с данными.

In [ ]:
cols = ["gmv", "platform_revenue", "logistics_cost", "promo_cost", "acquiring_cost", "return_cost", "reversed_revenue", "cm"]
unit = orders[cols].mean()
unit["реклама (1,1% GMV)"] = ADS * unit["gmv"]
unit["cm с рекламой"] = unit["cm"] + unit["реклама (1,1% GMV)"]
print(unit.round(0).to_string())
g = orders["gmv"].sum()
print(f"\nдоход платформы {orders['platform_revenue'].sum() / g:.1%} GMV · CM {orders['cm'].sum() / g:.1%} · CM с рекламой {orders['cm'].sum() / g + ADS:.1%}")
print(f"скидка 5% среднего чека = {0.05 * unit['gmv']:.0f} ₽ = {0.05 * unit['gmv'] / unit['cm с рекламой']:.0%} CM заказа")
ret = orders[orders["has_return"]]
print(f"заказов с возвратом {orders['has_return'].mean():.1%}; на возврате теряем {ret['return_cost'].mean():.0f} ₽ логистики "
      f"+ {ret['reversed_revenue'].mean():.0f} ₽ возвращённой комиссии")

**Вопросы.**
1. Где данные расходятся с паспортом и почему? *(Подсказка: какой строки дохода нет в `mart.orders`?)*
2. Сколько заказов с маржой 160 ₽ «съедает» один возврат?

## Часть 1. Когорты и удержание (20 мин)

**Когорта** — покупатели, сделавшие первый заказ в одном месяце. **Месяц жизни k** — дни с первого заказа, делённые на 30. Ячейка матрицы — доля покупателей когорты, сделавших хотя бы один заказ в месяце жизни k (N-месячное удержание по заказам).

**Недозревшая ячейка** — месяц, который ещё не прожили все покупатели когорты. Её нельзя сравнивать с остальными: она ниже не потому, что когорта хуже, а потому что время не прошло. Такие ячейки закрываем.

In [ ]:
first = orders.groupby("user_id")["order_ts"].min().rename("t1")
buyers = users.set_index("user_id").join(first, how="inner")          # только те, кто сделал заказ
buyers["cohort"] = buyers["t1"].dt.to_period("M")

o = orders.join(buyers[["t1", "cohort", "channel", "crm_segment", "reg_date"]], on="user_id", how="inner")
o["day"] = (o["order_ts"] - o["t1"]).dt.days        # дней с первого заказа
o["month"] = o["day"] // 30                         # месяц жизни покупателя: 0, 1, 2, …
o["value"] = o["cm"] + ADS * o["gmv"]               # маржа заказа с рекламой
print(f"покупателей {len(buyers):,} из {len(users):,} зарегистрированных ({len(buyers) / len(users):.0%})")

In [ ]:
def cohort_matrix(o, buyers, max_month=12):
    size = buyers.groupby("cohort").size()
    act = o[o["month"] <= max_month].groupby(["cohort", "month"])["user_id"].nunique().unstack()
    mat = act.div(size, axis=0)
    for c in mat.index:                       # закрываем недозревшие ячейки
        for k in mat.columns:
            if c.end_time + pd.Timedelta(days=30 * (k + 1)) > END:
                mat.loc[c, k] = np.nan
    return mat

mat = cohort_matrix(o, buyers)
show = mat.loc[:, 1:12]
fig, ax = plt.subplots(figsize=(12, 6.5))
im = ax.imshow(show.to_numpy(dtype=float), cmap="Blues", vmin=0.2, vmax=0.5, aspect="auto")
ax.set_yticks(range(len(show)), [str(c) for c in show.index]); ax.set_xticks(range(12), range(1, 13))
for i in range(show.shape[0]):
    for j in range(show.shape[1]):
        v = show.iat[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8, color="white" if v > 0.38 else "black")
ax.set_xlabel("месяц жизни k"); ax.set_ylabel("когорта первого заказа")
ax.set_title("Доля покупателей когорты с заказом в месяце k (пусто — ячейка не дозрела)")
plt.show()

**Вопросы.**
1. Выходит ли удержание на плато к 12-му месяцу? Что это значит для LTV «на всю жизнь»?
2. Какая строка выделяется? В каком месяце жизни это уже видно?
3. Почему нельзя сравнивать удержание 10-го месяца у когорты 2025-10 и 2025-03, если в строке 2025-10 эта ячейка есть, а дальше — пусто?

In [ ]:
mature = buyers[buyers["t1"] < "2025-09-01"].copy()     # когорты март–август 2025 прожили полный год
om = o[o["user_id"].isin(mature.index)]
ret_ch = (om[om["month"].between(1, 11)].groupby(["channel", "month"])["user_id"].nunique().unstack()
          .div(mature.groupby("channel").size(), axis=0))
fig, ax = plt.subplots(figsize=(10, 4.5))
for ch, row in ret_ch.iterrows():
    ax.plot(row.index, row.values, marker="o", lw=3 if ch in ("cpa", "organic") else 1.5, label=ch)
ax.set_xlabel("месяц жизни k"); ax.set_ylabel("доля с заказом в месяце k"); ax.legend(ncol=4)
ax.set_title("Удержание по каналам привлечения, когорты март–август 2025"); plt.show()
ret_ch[[1, 3, 6, 11]].round(3).sort_values(11, ascending=False)

**Удержание по чему?** «Вернулся» можно считать по сессиям (`mart.activity`) или по заказам. Деньги приносят заказы.

In [ ]:
act = con.execute("SELECT * FROM mart.activity").fetchdf()
a = act[act["user_id"].isin(mature.index)].join(mature["t1"], on="user_id")
a["month"] = (a["activity_date"] - a["t1"].dt.normalize()).dt.days // 30
ret_sess = a[a["month"].between(1, 11)].groupby("month")["user_id"].nunique() / len(mature)
ret_ord = om[om["month"].between(1, 11)].groupby("month")["user_id"].nunique() / len(mature)
print(pd.DataFrame({"сессия в месяце k": ret_sess, "заказ в месяце k": ret_ord}).loc[[1, 3, 6, 11]].round(3).T)

aug = act[act["activity_date"] >= "2026-08-01"]
dau, mau = aug.groupby("activity_date").size().mean(), aug["user_id"].nunique()
aug_o = orders[orders["order_ts"] >= "2026-08-01"]
print(f"\nавгуст 2026: DAU/MAU = {dau / mau:.2f}; заказов на покупателя в месяц {len(aug_o) / aug_o['user_id'].nunique():.2f}")

**Вопросы.**
1. Какой канал выглядит хуже всех в 1-м месяце, а какой — в 11-м? Совпадает ли ранжирование?
2. Удержание по сессиям выше, чем по заказам. Какое из двух вы поставите целевой метрикой команды удержания и почему?
3. DAU/MAU ≈ 0,26 при 2,3 заказа на покупателя в месяц. Это мало или нормально для продукта, которым пользуются раз в одну-две недели?
4. 🔵 Посчитайте **rolling retention** (доля покупателей с заказом в месяце k *или позже*) и сравните с N-месячным. Почему rolling всегда выше и почему у молодых когорт он особенно обманчив?

## Часть 2. Провал когорты «Чёрной пятницы» (20 мин)

Распродажа шла с 20 ноября по 1 декабря 2025 года. Сравним тех, кто **зарегистрировался** в эти дни, с соседями до и после.
Чтобы не сравнивать недозревшее с дозревшим, каждую долю считаем только по покупателям, которые уже прожили нужный срок.

In [ ]:
U = users.set_index("user_id").join(first)
U["buyer"] = U["t1"].notna()
reg_day = U[(U["reg_date"] >= "2025-10-15") & (U["reg_date"] < "2025-12-20")].groupby(U["reg_date"].dt.date).size()
reg_day.plot(figsize=(10, 3), title="Регистрации в день, октябрь–декабрь 2025"); plt.show()

WIN = {"до: 01.10–14.11": ("2025-10-01", "2025-11-15"),
       "Чёрная пятница: 20.11–01.12": ("2025-11-20", "2025-12-02"),
       "после: 08.12–19.01": ("2025-12-08", "2026-01-20")}

def lived(b, days):            # покупатели, которые уже прожили days дней с первого заказа
    return b[b["t1"] + pd.Timedelta(days=days) <= END]

def cohort_card(lo, hi):
    reg = U[(U["reg_date"] >= lo) & (U["reg_date"] < hi)]
    b = reg[reg["buyer"]]
    ob = o[o["user_id"].isin(b.index)]
    row = {"регистраций в день": len(reg) / (pd.Timestamp(hi) - pd.Timestamp(lo)).days,
           "доля ставших покупателями": len(b) / len(reg),
           "доля промо-охотников": (b["crm_segment"] == "promo_hunter").mean(),
           "доля CPA и соцсетей": b["channel"].isin(["cpa", "social"]).mean(),
           "CM первого заказа, ₽": ob.loc[ob["order_seq"] == 1, "value"].mean()}
    for k in (1, 3, 6):
        bk = lived(b, 30 * (k + 1))
        row[f"заказ в {k}-м месяце"] = ob[(ob["month"] == k) & ob["user_id"].isin(bk.index)]["user_id"].nunique() / len(bk)
    b180 = lived(b, 180)
    row["маржа 180 дней на покупателя, ₽"] = ob[(ob["day"] < 180) & ob["user_id"].isin(b180.index)]["value"].sum() / len(b180)
    return pd.Series(row)

bf = pd.DataFrame({k: cohort_card(*v) for k, v in WIN.items()})
bf.round(3)

**Две гипотезы, почему когорта хуже:**
- **(а) состав** — распродажа привела больше промо-охотников и трафика из CPA-сетей и соцсетей;
- **(б) механика** — сама распродажа: купили «впрок» и по скидке, привычки к «Лукошку» не сложилось.

Сравним покупателей **одного и того же сегмента** до и во время распродажи.

In [ ]:
rows = []
for name, (lo, hi) in list(WIN.items())[:2]:
    b = U[(U["reg_date"] >= lo) & (U["reg_date"] < hi) & U["buyer"]]
    for seg, bs in b.groupby("crm_segment"):
        b120, b180 = lived(bs, 120), lived(bs, 180)
        ob = o[o["user_id"].isin(bs.index)]
        rows.append({"окно": name, "сегмент": seg, "покупателей": len(bs),
                     "заказ в 3-м месяце": ob[(ob["month"] == 3) & ob["user_id"].isin(b120.index)]["user_id"].nunique() / len(b120),
                     "маржа 180 дней, ₽": ob[(ob["day"] < 180) & ob["user_id"].isin(b180.index)]["value"].sum() / len(b180)})
within = pd.DataFrame(rows).set_index(["сегмент", "окно"]).sort_index()
within.round(3)

**Вопросы.**
1. Во сколько раз выросли регистрации в дни распродажи и что стало с маржой 180 дней на покупателя?
2. Что объясняет провал — состав, механика или оба? Опирайтесь на сравнение внутри сегментов.
3. Почему в отчёте «удержание по всем покупателям» этот провал размазан и почти не виден? *(Вспомните матрицу: ноябрьская строка.)*
4. Что вы предложите изменить в распродаже следующего года — и какую метрику поставите ей целевой вместо «новых регистраций»?
5. 🔵 Разложите разницу маржи 180 дней на **эффект состава** (другие доли сегментов при «старой» марже сегментов) и **эффект внутри сегментов**. Какая часть больше?

## Часть 3. Юнит «в среднем → в сегментах» (25 мин)

LTV12 = маржа (CM + реклама) за 365 дней с первого заказа. Когорты март–август 2025 — у них полный год.

In [ ]:
o12 = om[om["day"] < 365]
mature["ltv12"] = o12.groupby("user_id")["value"].sum().reindex(mature.index, fill_value=0)
x = mature["ltv12"]
top10 = x.nlargest(len(x) // 10).sum() / x.sum()
print(f"LTV12 на покупателя: среднее {x.mean():,.0f} ₽, медиана {x.median():,.0f} ₽; "
      f"топ-10% покупателей дают {top10:.0%} маржи; LTV12 < 0 у {(x < 0).mean():.0%}")
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(x.clip(-500, 10000), bins=105, color="#5B7DB1")
for v, lab, c in [(x.median(), "медиана", "black"), (x.mean(), "среднее", "#E97132")]:
    ax.axvline(v, color=c, lw=2); ax.text(v + 80, ax.get_ylim()[1] * 0.85, f"{lab} {v:,.0f} ₽", color=c)
ax.set_xlabel("LTV12, ₽ (обрезано на 10 000)"); ax.set_title("Распределение LTV12 по покупателям"); plt.show()

In [ ]:
CAC = pd.Series({"organic": 0, "ctx_brand": 380, "ctx_category": 950, "social": 1150,
                 "cpa": 1400, "influencer": 1900})            # паспорт, §8; у ТВ CAC не известен

def payback_month(cum, cac):
    """Номер месяца (1 — месяц первого заказа), в котором накопленная маржа догнала CAC."""
    if pd.isna(cac) or cac == 0:
        return np.nan
    hit = np.flatnonzero(cum.to_numpy() >= cac)
    return hit[0] + 1 if len(hit) else np.inf

def cum_margin(by):
    """Накопленная маржа на покупателя по месяцам жизни 0…11 в разрезе by."""
    return (o12.groupby(by + ["month"])["value"].sum().unstack().cumsum(axis=1)
            .div(mature.groupby(by).size(), axis=0))

cum_ch = cum_margin(["channel"])
tab = mature.groupby("channel")["ltv12"].agg(покупателей="size", LTV12="mean", медиана="median")
tab["CAC"] = CAC
tab["LTV/CAC"] = tab["LTV12"] / tab["CAC"]
tab["payback, мес"] = [payback_month(cum_ch.loc[c], CAC.get(c)) for c in tab.index]
blended = (tab["LTV12"] * tab["покупателей"]).sum() / tab["покупателей"].sum()
print(f"в среднем по всем: LTV12 {blended:,.0f} ₽, при blended CAC 640 ₽ (паспорт) LTV/CAC = {blended / 640:.1f}")
tab.round(2).sort_values("LTV/CAC", ascending=False)

**Вопрос «в среднем».** CPA-сети: LTV/CAC < 1, за 12 месяцев канал не окупается. Вывод «в среднем» — выключить CPA. Теперь разрежем по CRM-сегменту.

In [ ]:
seg = mature[mature["channel"].isin(["cpa", "social", "ctx_category"])]
cum_seg = cum_margin(["channel", "crm_segment"])
st = seg.groupby(["channel", "crm_segment"])["ltv12"].agg(покупателей="size", LTV12="mean")
st["CAC"] = [CAC[c] for c, _ in st.index]
st["LTV/CAC"] = st["LTV12"] / st["CAC"]
st["payback, мес"] = [payback_month(cum_seg.loc[i], st.loc[i, "CAC"]) for i in st.index]
print(st.round(2))

fig, ax = plt.subplots(figsize=(10, 4))
for s, col in [("regular", "#0E2841"), ("promo_hunter", "#E97132")]:
    y = cum_seg.loc[("cpa", s)].values
    ax.plot(range(1, len(y) + 1), y, marker="o", color=col, lw=3, label=f"CPA, {s}")
ax.axhline(CAC["cpa"], color="grey", ls="--"); ax.text(1, CAC["cpa"] + 40, "CAC CPA = 1 400 ₽")
ax.set_xlabel("месяц жизни (1 — месяц первого заказа)"); ax.set_ylabel("накопленная маржа, ₽")
ax.set_title("CPA-сети: накопленная маржа на покупателя"); ax.legend(); plt.show()

In [ ]:
# Различим ли сегмент заранее? Промо-расходы на заказ по его номеру
pc = o.assign(n=o["order_seq"].clip(upper=4)).groupby(["crm_segment", "n"])["promo_cost"].mean().unstack().round(0)
pc.columns = ["1-й заказ", "2-й", "3-й", "4-й и дальше"]
pc

**Вопросы.**
1. Как разворачивается решение по CPA-сетям, если смотреть в сегментах? Сформулируйте его одной фразой с цифрами LTV/CAC и payback.
2. Сеть не умеет фильтровать промо-охотников. Что можно сделать с условиями закупки трафика? *(Подсказка: на каком заказе сегмент становится различим и за что можно платить сети.)*
3. Что изменится в решении по контексту по категориям и по соцсетям?
4. Среднее LTV12 почти втрое выше медианы. Какую из двух цифр вы бы использовали для решения о CAC канала — и для какого решения нужна другая?
5. 🔵 Постройте бутстреп-интервал (95%) для LTV/CAC обычных покупателей и промо-охотников CPA. Отличается ли LTV/CAC обычных от 1 при ~340 покупателях?

In [ ]:
# 🔵 глубокий трек: бутстреп LTV/CAC внутри CPA (сначала попробуйте сами)
rng = np.random.default_rng(0)
for s in ["regular", "promo_hunter"]:
    v = mature[(mature["channel"] == "cpa") & (mature["crm_segment"] == s)]["ltv12"].to_numpy()
    bs = rng.choice(v, (2000, len(v))).mean(axis=1) / CAC["cpa"]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"CPA, {s:12s}: LTV/CAC = {v.mean() / CAC['cpa']:.2f}, 95% бутстреп [{lo:.2f}; {hi:.2f}], n = {len(v)}")

## Часть 4. Простой прогноз LTV и стресс-тест (20 мин)

У молодых когорт LTV12 ещё не наблюдён. Прогнозируем его по первым 90 дням и проверяем прогноз на когортах, где ответ уже известен:
обучаемся на когортах март–май 2025, проверяем на июнь–август 2025.

- **A: когортный множитель** — LTV12 = маржа за 90 дней × k, где k = сумма LTV12 / сумма маржи 90 дней на обучении.
- **B: регрессия** — LTV12 по марже и числу заказов за 90 дней, флагу промо-охотника и каналу.

In [ ]:
import statsmodels.formula.api as smf

def features(b):
    ob = o[o["user_id"].isin(b.index) & (o["day"] < 90)]
    f = pd.DataFrame(index=b.index)
    f["cm90"] = ob.groupby("user_id")["value"].sum().reindex(f.index, fill_value=0)
    f["n90"] = ob.groupby("user_id").size().reindex(f.index, fill_value=0)
    f["promo"] = (b["crm_segment"] == "promo_hunter").astype(int)
    f["channel"], f["segment"] = b["channel"], b["crm_segment"]
    return f

F = features(mature).assign(ltv12=mature["ltv12"])
train, test = F[mature["t1"] < "2025-06-01"], F[mature["t1"] >= "2025-06-01"].copy()
k = train["ltv12"].sum() / train["cm90"].sum()
model = smf.ols("ltv12 ~ cm90 + n90 + promo + C(channel)", train).fit()
test["A: множитель"] = test["cm90"] * k
test["B: регрессия"] = model.predict(test)
print(f"множитель k = {k:.2f}; обучение {len(train):,}, проверка {len(test):,} покупателей")

def errors(t, col):
    e = t[col] - t["ltv12"]
    top, low = t["ltv12"] >= t["ltv12"].quantile(0.99), t["ltv12"] <= t["ltv12"].quantile(0.30)
    return pd.Series({"MAE, ₽": e.abs().mean(), "смещение в среднем, ₽": e.mean(),
                      "смещение у топ-1%, ₽": e[top].mean(), "смещение у нижних 30%, ₽": e[low].mean()})

pd.DataFrame({c: errors(test, c) for c in ["A: множитель", "B: регрессия"]}).round(0)

In [ ]:
by_seg = test.groupby(["channel", "segment"])[["ltv12", "A: множитель", "B: регрессия"]].mean()
for c in ["A: множитель", "B: регрессия"]:
    by_seg[c + ", ошибка"] = (by_seg[c] / by_seg["ltv12"] - 1).map("{:+.0%}".format)
by_seg.loc[["cpa", "social", "ctx_category", "organic"]].round(0)

**Вопросы.**
1. В среднем множитель A почти не ошибается. Где он ошибается так, что это меняет решение?
2. Обе модели сильно занижают топ-1% покупателей. Важно ли это для решения о закупке трафика по каналу? А для решения «кому дать персонального менеджера»?
3. Почему MAE по отдельному покупателю — плохая метрика для этой задачи? Какую ошибку вы бы мерили вместо неё?

### Прогноз в таблицу экономики и стресс-тест «CAC +20%»

Берём молодые когорты (первый заказ февраль–май 2026: у всех есть 90 дней), прогнозируем LTV12 моделью B и считаем, что будет, если аукцион подорожает на 20%.

In [ ]:
young = buyers[(buyers["t1"] >= "2026-02-01") & (buyers["t1"] < "2026-06-01")]
Fy = features(young)
Fy["LTV12 прогноз"] = model.predict(Fy)
econ = Fy[Fy["channel"].isin(CAC.index) & (Fy["channel"] != "organic")].groupby(["channel", "segment"]).agg(
    покупателей=("LTV12 прогноз", "size"), LTV12_прогноз=("LTV12 прогноз", "mean"))
econ["CAC"] = [CAC[c] for c, _ in econ.index]
econ["LTV/CAC"] = econ["LTV12_прогноз"] / econ["CAC"]
econ["LTV/CAC при CAC +20%"] = econ["LTV/CAC"] / 1.2
econ["факт LTV/CAC (когорты 2025)"] = [mature[(mature["channel"] == c) & (mature["crm_segment"] == s)]["ltv12"].mean() / CAC[c]
                                       for c, s in econ.index]
econ["вердикт при +20%"] = np.where(econ["LTV/CAC при CAC +20%"] >= 1, "окупается",
                             np.where(econ["LTV/CAC"] >= 1, "выпадает", "не окупался и так"))
econ.round(2)

In [ ]:
# payback по фактическим кривым когорт 2025 при текущем CAC и при CAC +20%
pb = pd.DataFrame([{"канал": c, "сегмент": s,
                    "payback, мес": payback_month(cum_seg.loc[(c, s)], CAC[c]),
                    "payback при +20%, мес": payback_month(cum_seg.loc[(c, s)], 1.2 * CAC[c])}
                   for c, s in cum_seg.index if c in CAC.index and c != "organic"])
print("inf — не окупается за 12 месяцев")
pb.set_index(["канал", "сегмент"])

**Вопросы к части 4** (материал для memo):
1. Какие сегменты выпадают из окупаемости при CAC +20%? Сколько месяцев добавляется к payback обычных покупателей CPA?
2. На сколько должен вырасти LTV12, чтобы сохранить текущую закупку трафика? Чьи это рычаги — маркетинга или продукта?
3. Прогноз LTV молодых когорт выше, чем факт когорт 2025. Это продукт стал лучше или модель ошибается? Как это проверить и что делать с решением, пока не проверили?
4. 🔵 Замените модель B на градиентный бустинг (`sklearn.ensemble.HistGradientBoostingRegressor`). Изменилась ли **таблица решений** (колонка «вердикт»)? Если нет — стоило ли улучшать модель?

## Итог семинара: memo директору по маркетингу (полстраницы)

1. Что не так с выводом «CPA-сети не окупаются, выключаем» — с цифрами LTV/CAC и payback в сегментах.
2. Что делать с когортой распродажи и какую метрику ставить распродаже следующего года.
3. Какие каналы и сегменты выпадут, если аукцион подорожает на 20%, и какой рычаг продукта это компенсирует.